# 15.07 · Benchmarking cost and latency

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 2 h |
| 📋 Prerequisites | Lessons 15.01–15.06; Module 13 (latency percentiles, cost per request). |
| 🔑 Keys | optional `TYPESAFE_API_KEY` (live Jev latency) · optional `GROQ_API_KEY` (LLM baseline, 20 calls) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (no network, free) · **⚖️** Jev: live with `TYPESAFE_API_KEY` (paid, fractions of a cent), otherwise replayed from recordings · **💸** calls Groq (uses your free-tier quota).
> Without keys, ⚖️ cells replay recorded answers, 💸 cells print `⏭️ Skipped`, and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Benchmark** a decision call honestly: warm-up, p50 / p95 instead of a single average, and a label saying what was measured.
2. **Tell apart** client overhead (replay), network + model time (live), and throughput from concurrency.
3. **Price** decisions per 1,000 calls from measured token usage and project the monthly cost at your traffic.
4. **Decide** between Jev, a small LLM and a large LLM for one decision, from your own numbers.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [TypeSafe pricing](https://typesafe.ai/pricing) | Per-token price, what usage is billed |
| [Groq: models and pricing](https://console.groq.com/docs/models) | The LLM baseline's price and limits |
| [Groq: rate limits](https://console.groq.com/docs/rate-limits) | Why the free tier caps a benchmark's concurrency |
| [LangChain: `batch` and `max_concurrency`](https://docs.langchain.com/oss/python/langchain/runnables) | Throughput with a Runnable |

> **Why this matters at work.** "Decision models are faster and cheaper" is a vendor's claim. Before Orbita moves triage off an LLM, someone has to show the p95 latency and the monthly bill, measured on Orbita's tickets, with the method written down. This lesson builds that worksheet.

> ⚠️ **Read the `source` column.** In replay mode, Jev's "latency" is how long *the client* takes to build a request and parse a recorded answer, with no network and no model. It's useful for seeing the harness's overhead and nothing else. Live numbers need `TYPESAFE_API_KEY` (Part 4).

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-typesafe[experimental]==0.0.1a3" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-typesafe", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY', 'GROQ_API_KEY')
LANGSMITH_PROJECT = 'm15-benchmarking-decisions'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

from mastery.decisions import decision_cost, provenance
from mastery.decisions.bench import benchmark, llm_cost_per_1k, monthly_cost, percentile
from mastery.decisions.orbita import lab_tickets, queue_questions

TICKETS = lab_tickets()[:40]                   # 40 of Lesson 15.02's lab tickets (all recorded)
request = lambda t: {"state": t["text"], "questions": queue_questions()}
print(len(TICKETS), "tickets · questions:", list(queue_questions()))
print("percentile demo:", percentile([10, 12, 11, 13, 250], 50), "ms p50 ·", percentile([10, 12, 11, 13, 250], 95), "ms p95")

**You should now see** 40 tickets, the two questions from Lesson 15.02 (`queue`, `is_outage`), and a small percentile demo. On `[10, 12, 11, 13, 250]` the p50 is `12` and the p95 is `250`. One slow call ruins the p95 and barely moves the median, which is why a mean alone hides the tail your users feel.

---
## Part 1 · The harness on replay: client overhead only

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ Time Jev one call at a time (1 warm-up call), and price each call from its usage.
src = "replay: client overhead only" if JEV.mode == "replay" else "live: network + model"
seq = benchmark("Jev, sequential", lambda t: jev.invoke(request(t)), TICKETS, cost_of=decision_cost, source=src)

import time
start = time.perf_counter()
answers = jev.batch([request(t) for t in TICKETS], config={"max_concurrency": 8})
wall = time.perf_counter() - start
table([seq.row()], ["setup", "n", "p50 ms", "p95 ms", "mean ms", "$ per 1k", "source"])
print(f"\nbatch, max_concurrency=8: {len(answers)} answers in {wall * 1000:.0f} ms wall "
      f"({len(answers) / wall:.0f} decisions/s) · answers: {provenance(answers[0])}")

**You should now see** a row with **sub-millisecond** p50 and p95 in replay, the source **`replay: client overhead only`**, and a `$ per 1k` of about `0.01` (priced from the recorded token usage). Then a batch line with the wall time for all 40.

- Replay p50 tells you the classifier adds a few ms of Python per call. Don't quote it as Jev's latency.
- The cost **is** meaningful offline: it's the usage tokens × the price table in `mastery.pricing`. In replay the usage is synthetic (a characters ÷ 4 estimate), so it's in the right range, not exact.
- `batch(..., max_concurrency=8)` improves **throughput** (decisions per second), not the latency of any single call. A ticket queue cares about throughput, and a user waiting on a chatbot cares about latency.

---
## Part 2 · Cost per 1,000 decisions and per month

In [ ]:
# 🧪 Price the same decision with Jev (measured usage) and two Groq LLMs (estimated tokens; Part 3 measures them).
in_tokens = sum(len(t["text"]) for t in TICKETS) / len(TICKETS) / 4 + 350   # ticket + system prompt + JSON schema (estimate)
out_tokens = 40                                                                # a short JSON answer
setups = [("Jev (jev-1.13.0)", seq.usd_per_1k, "measured usage" if JEV.mode == "live" else "replay usage (synthetic)"),
          ("Groq gpt-oss-20b", llm_cost_per_1k("groq:openai/gpt-oss-20b", in_tokens, out_tokens), f"estimate: {in_tokens:.0f} in / {out_tokens} out"),
          ("Groq gpt-oss-120b", llm_cost_per_1k("groq:openai/gpt-oss-120b", in_tokens, out_tokens), f"estimate: {in_tokens:.0f} in / {out_tokens} out")]
rows = [{"setup": name, "$ per 1k": f"{usd:.4f}", "10k/day": f"${monthly_cost(usd, 1e4):,.2f}/mo",
         "1M/day": f"${monthly_cost(usd, 1e6):,.0f}/mo", "tokens": how} for name, usd, how in setups]
table(rows, ["setup", "$ per 1k", "10k/day", "1M/day", "tokens"])

**You should now see** three rows. Jev costs about a cent per 1,000 decisions. The small Groq model costs roughly 4× that, and the large one about 8×. At 1M decisions a day the gap is the difference between a rounding error and a line item.

- These are **paid-tier list prices** from `mastery.pricing` (verified 2026-10-16). Groq's free tier costs $0 but caps requests per minute, so it can't carry production traffic.
- The LLM token counts are an **estimate** until Part 3 replaces them with measured usage. The 350-token overhead (system prompt + schema) is a guess, so check it there.
- Cost isn't the whole decision: Lesson 15.02's accuracy and calibration, and the LLM's ability to explain itself (15.06), belong on the same worksheet.

---
## Part 3 · The LLM baseline, measured

In [ ]:
# 💸 Groq's fast model answering the SAME questions in Jev's shape (llm_decider), 20 tickets one at a time (~10K tokens).
from mastery.decisions import llm_decider

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    from mastery.models import get_model
    llm = llm_decider(get_model("fast", temperature=0))
    usage = []
    def run_llm(t):
        r = llm.invoke(request(t))
        usage.append((r.usage.input_tokens or 0, r.usage.output_tokens or 0))
        return r
    llm_bench = benchmark(f"Groq {settings.model_fast}", run_llm, TICKETS[:20], source="live: network + model")
    mean_in = sum(u[0] for u in usage) / len(usage); mean_out = sum(u[1] for u in usage) / len(usage)
    llm_usd = llm_cost_per_1k("groq:" + settings.model_fast, mean_in, mean_out)
    row = llm_bench.row() | {"$ per 1k": "n/a" if llm_usd is None else f"{llm_usd:.4f}"}
    table([seq.row(), row], ["setup", "n", "p50 ms", "p95 ms", "mean ms", "$ per 1k", "source"])
    print(f"\nmeasured LLM tokens per call: {mean_in:.0f} in / {mean_out:.0f} out (Part 2 estimated {in_tokens:.0f} / {out_tokens})")

**You should now see** (with a key) Groq's row next to Jev's. Groq is fast for an LLM, typically a few hundred ms p50, with a p95 that can jump when the free tier queues you. The measured tokens per call replace Part 2's estimate. Without a key it prints `⏭️ Skipped`.

- The two rows are only comparable when **both** sources say `live`. In replay mode, the table puts a real network call next to a local lookup, and that's the mistake this lesson is about.
- 20 calls give a rough p95 at best (it's the 19th of 20 values). For a real decision, run a few hundred calls spread over a day.

---
## Part 4 · Live Jev latency

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ Only meaningful with TYPESAFE_API_KEY: Jev over the network, sequential, then batched.
if JEV.mode != "live":
    print("⏭️ Replay mode: live Jev latency needs TYPESAFE_API_KEY. Part 1's numbers are client overhead only.")
else:
    live = benchmark("Jev live, sequential", lambda t: jev.invoke(request(t)), TICKETS, cost_of=decision_cost,
                     source="live: network + model")
    table([live.row()], ["setup", "n", "p50 ms", "p95 ms", "mean ms", "$ per 1k", "source"])

**You should now see**, in replay, `⏭️ Replay mode: …`. With a funded key, you get a row with real network + model latency. TypeSafe describes Jev as answering in milliseconds, so expect tens of ms plus your round-trip time to the API (the region matters a lot from Bogotá). **Measure it, then write it down.** Don't plan on the vendor's figure.

---
## Part 5 · The decision worksheet

| Question | Where the number comes from |
|---|---|
| Is it accurate enough on *our* data? | Lesson 15.02 (accuracy, calibration on a labeled split) |
| What does the tail latency look like? | Parts 3–4, p95 with `source = live` |
| What will it cost at our volume? | Part 2 with Part 3's measured tokens |
| What happens on the cases it's unsure about? | Lessons 15.03–15.04 (fallback, escalation share × LLM cost) |
| Does anyone need an explanation? | Lesson 15.06: if yes, an LLM is still in the loop somewhere |

Lesson 15.04's hybrid is the usual answer: Jev on every item, the LLM only on the escalated share. Its cost is `Jev $/1k + escalation rate × LLM $/1k`. Exercise 2 computes it.

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| "Jev answers in 1 ms" | Replay numbers quoted as latency | Read the `source` column; measure live |
| First call much slower than the rest | Connection setup, imports | Warm-up call (`warmup=1`), report p50 / p95 |
| p95 jumps run to run | Too few samples, free-tier queueing | More calls, spread over time; paid tier for benchmarks |
| `429` during the LLM benchmark | Groq free-tier rate limit | Sequential calls, smaller sample, retry with backoff |
| Cost per 1k is `n/a` | Model not in `mastery.pricing` | Add the model's price, with its source and date |

> ⚠️ **Production note: benchmark where you deploy.** (Parts 3–4)
> Latency from a laptop in Bogotá to a US region isn't latency from a server in that region. Re-run the harness from your production environment before you commit to an SLA.

## 🏋️ Exercises

1. **(Easy)** Run Part 1 with `max_concurrency=1` and `=16`. Which number changes, the per-call latency or the throughput?
2. **(Medium)** With Lesson 15.04's escalation rate (27 of 234 tickets), compute the hybrid's `$ per 1k` from Part 2's rows. How does it compare with "LLM on every ticket"?
3. **(Medium)** Add a `p99_ms` field to `BenchResult` (through `percentile(times, 99)`). How many samples do you need before p99 is more than "the slowest call"?
4. **(Stretch, needs keys)** Run Parts 3–4 live at three times of day and plot p50 / p95 per setup.

<details>
<summary><b>Show solutions</b></summary>

**1.** Throughput: decisions per second rises with concurrency until the API (or your rate limit) pushes back. Each call's own latency stays about the same, or grows slightly under load.

**2.** `hybrid = jev + 27 / 234 × llm`. With Part 2's offline numbers: `0.0099 + 0.115 × 0.0395 ≈ 0.0145` per 1k, about 37% of the small LLM on every ticket (`0.0395`), and about 18% of the large one. The Jev share is nearly free, so the escalation rate drives the bill.

**3.** With nearest-rank, p99 of n samples is the slowest one until n ≥ 100. Below that, report p95 and say so.

**4.** Collect `BenchResult.row()` per run with a timestamp and plot with matplotlib. Expect the LLM's p95 to move more across the day than Jev's.
</details>

## 🔎 Docs drill (5 min)

Does TypeSafe bill output tokens, and where does the response say how many tokens a request used?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open **TypeSafe → Pricing** (linked above) and the API reference for the classify response.
2. Pricing is per **input** token ($0.042 per 1M for Jev as of 2026-10-16); `mastery.pricing` records output at $0.
3. The response's **`usage`** (`input_tokens`, `output_tokens`) is what `decision_cost` multiplies by the price.

</details>

## ✅ Checkpoint

- [ ] I benchmarked with a warm-up and reported p50 / p95, with a `source` label.
- [ ] I can explain why replay latency isn't Jev's latency, and why batch raises throughput, not speed.
- [ ] I priced a decision per 1,000 calls and per month at our volume.
- [ ] I filled in the decision worksheet for one Orbita decision.
- [ ] Restart & Run All passes with no keys; outputs cleared before saving.

## 🔎 Research checkpoint

Re-check TypeSafe's and Groq's prices (they move), and update `mastery.pricing` with the source and date if they changed. Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- A benchmark is warm-up, percentiles, a labeled source and a written method. One average from a replay is none of those.
- Cost per decision comes from measured usage × a dated price table. Monthly cost at your volume is what decides.
- Jev, a small LLM and a large LLM aren't rivals: the hybrid from Lesson 15.04 uses each where it's cheapest.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 15 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)